# Python Bridge for 12a

## A Short Code Refresher

Use this optional notebook alongside 12a to practice a chronological split, scikit-learn regression, component recombination, and aligned forecast-error calculations. Allow about 20 minutes. Twenty invented quarterly observations keep the tables small and remove the need for external data.

The lecture uses a longer monthly generation series. Here a straight trend and repeated quarterly effects make the Python steps inspectable. The planted pattern is a code example, not evidence about which forecasting method will win on real data.

## References

Scikit-learn documents [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html), including `fit`, `predict`, and fitted attributes. Pandas documents [DataFrame.rsub](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rsub.html), used to align observed values with prediction columns. [Downey's prediction example](https://allendowney.github.io/ThinkStats/chap12.html#prediction) supplies the component-forecast approach used in the lecture.

## Reserve the Last Four Quarters

Create a complete quarterly series with a linear increase and effects of −2, 3, 1, and −2 units across the four quarters. `.iloc` slices rows by position. The final four rows form the test series; every fitted quantity below uses earlier rows only.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.seasonal import seasonal_decompose

dates = pd.period_range("2020Q1", periods=20, freq="Q")
values = 100 + 0.5 * np.arange(20) + np.tile([-2, 3, 1, -2], 5)
series = pd.Series(values, index=dates, name="Observed")
horizon = 4
training = series.iloc[:-horizon]
test = series.iloc[-horizon:]
assert training.index.max() < test.index.min()
display(pd.DataFrame({"Training": training, "Test": test}).tail(7))

Sixteen observations train the models and four are withheld. The missing cells in the display identify which partition owns each date; they are not missing source measurements.

## Build a Two-Dimensional Predictor Table

Give each quarter an elapsed-time coordinate and retain the dates as labels. `.loc` selects the predictor rows matching each partition. Knowing future calendar dates does not reveal their observed values.

In [ ]:
X = pd.DataFrame({"Elapsed quarter": np.arange(len(series))}, index=series.index)
X_train = X.loc[training.index]
X_test = X.loc[test.index]
print("Training predictor shape:", X_train.shape)
print("Training response shape:", training.shape)
display(X_test)

`X_train` has shape `(16, 1)` and the response Series has shape `(16,)`. The test predictor continues at 16 rather than restarting at zero. Elapsed quarter counts time since the beginning; calendar quarter identifies a seasonal position.

## Fit Once and Predict Supplied Rows

`LinearRegression()` constructs the estimator. `fit(X, y)` learns its intercept and coefficient from training values. `predict(X_test)` applies those learned values to future elapsed quarters without fitting again.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, training)
baseline_forecast = pd.Series(baseline.predict(X_test), index=test.index, name="Time only")
display(pd.Series({"Intercept": baseline.intercept_, "Slope": baseline.coef_[0]}))
display(baseline_forecast)

The trailing underscores mark fitted attributes. The forecast is a Series with the test dates restored. The fitted line summarizes the original observations; it does not separately estimate their seasonal component.

## Fit a Line to the Training Trend-Cycle

Decompose only the training series. Combine its estimated trend-cycle with the existing elapsed-time predictor, then remove rows lacking a centered trend estimate. Selecting the predictor with a list of column names keeps it two-dimensional.

In [ ]:
decomposition = seasonal_decompose(training, model="additive", period=4)
trend_data = pd.DataFrame(
    {"Elapsed quarter": X_train["Elapsed quarter"], "Trend-cycle": decomposition.trend}
).dropna()
trend_model = LinearRegression()
trend_model.fit(trend_data[["Elapsed quarter"]], trend_data["Trend-cycle"])
trend_forecast = pd.Series(trend_model.predict(X_test), index=test.index)
display(trend_data.head())

The dropped endpoint rows retain their original elapsed-time values. We have not renumbered the remaining observations or used test measurements to estimate the trend-cycle.

## Look Up and Add Seasonal Effects

Group the training seasonal component by calendar quarter. Look up one effect for each forecast quarter, then attach the forecast dates before adding Series. Without that last step, pandas would try to align quarter-number labels with date labels.

In [ ]:
quarter_effects = decomposition.seasonal.groupby(training.index.quarter).mean()
effect_values = quarter_effects.loc[test.index.quarter].to_numpy()
seasonal_forecast = pd.Series(effect_values, index=test.index)
component_forecast = (trend_forecast + seasonal_forecast).rename("Trend plus season")
display(quarter_effects.rename("Effect"))
display(
    pd.DataFrame(
        {"Trend": trend_forecast, "Seasonal": seasonal_forecast, "Combined": component_forecast}
    )
)

The four effects reproduce the constructed pattern: −2, 3, 1, and −2. The trend forecast supplies the level and slope at each future quarter; the lookup adds that quarter's seasonal effect.

## Align Errors With the Observations

Collect both forecasts in columns. `predictions.rsub(test, axis="index")` computes observed minus predicted for each column, aligning the observed Series by date. Check the alignment and missing values before averaging.

In [ ]:
predictions = pd.concat([baseline_forecast, component_forecast], axis=1)
assert predictions.index.equals(test.index)
assert predictions.notna().all().all()
errors = predictions.rsub(test, axis="index")
scores = pd.DataFrame(
    {"MAE (units)": errors.abs().mean(), "RMSE (units)": np.sqrt(errors.pow(2).mean())}
)
display(errors.round(3))
display(scores.round(3))

The means run down the rows, producing one score per method. The component forecast recovers this deliberately simple pattern up to numerical precision. That verifies the calculation; a real comparison needs unseen data whose behavior was not constructed to match a model.

Plot both forecasts with the same observations and units. Converting quarters to timestamps supplies positions Matplotlib can draw.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
ax.plot(series.index.to_timestamp(), series, "o-", color="0.4", label="Observed")
for name in predictions:
    ax.plot(test.index.to_timestamp(), predictions[name], "o--", label=name)
ax.axvline(test.index[0].start_time, color="black", linestyle=":", label="Test begins")
ax.set(title="One Forecast Origin, Two Procedures", xlabel="Quarter start", ylabel="Units")
ax.legend()
plt.show()

## Try One Small Modification

Change `horizon = 4` to `horizon = 8` in the first cell, then restart and run all cells. Before doing so, identify which values the change should affect: the split, fitted models, and forecast dates all depend on the horizon. Updating only the test slice would leave stale fitted objects.

The seasonal period stays four because the observations remain quarterly. The holdout length and observations per seasonal cycle are different settings.

## Ready for 12a

Return to the lecture when you can preserve a chronological split, pass the right input shapes to `fit` and `predict`, repeat seasonal effects on future dates, and calculate date-aligned errors. The lecture extends the same workflow to the packaged STL forecast and a longer horizon.

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

For course materials, attribution, and licensing information, see the [INSY 7130 course-materials README](../../README.md).